# Fine-Tuning LLaMA 3.1-8B-Instruct on Bengali Empathetic Conversations

This notebook fine-tunes LLaMA 3.1-8B-Instruct using LoRA/Unsloth for efficient training on Kaggle free GPUs.

## Requirements:
- Full-sequence tokenization (no sequence length reduction)
- LoRA or Unsloth for parameter-efficient fine-tuning
- Evaluation with Perplexity, BLEU, ROUGE
- Experiment logging


## 1. Setup and Installation


In [59]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

%pip install -q "transformers>=4.40.0,<5.0.0"
%pip install -q "peft>=0.8.0"
%pip install -q "datasets>=2.14.0"
%pip install -q "accelerate>=0.27.0"
%pip install -q "bitsandbytes>=0.41.0"
%pip install -q "sacrebleu>=2.3.0"
%pip install -q "rouge-score>=0.1.2"
%pip install -q "nltk>=3.8.0"
%pip install -q "trl>=0.7.0"



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [60]:
%pip install -q unsloth

Note: you may need to restart the kernel to use updated packages.


In [361]:
import transformers
import torch
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

Transformers: 4.57.1
CUDA: True


In [362]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import torch
from torch.utils.data import Dataset, DataLoader

from unsloth import FastLanguageModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset, Dataset as HFDataset
import nltk
from sacrebleu import BLEU
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n" + "=" * 60)
print("KAGGLE INPUT DIRECTORIES")
print("=" * 60)
kaggle_input = "/kaggle/input"
if os.path.exists(kaggle_input):
    print(f"\nContents of {kaggle_input}:")
    for item in os.listdir(kaggle_input):
        item_path = os.path.join(kaggle_input, item)
        if os.path.isdir(item_path):
            print(f"  📁 {item}/")
            try:
                subdirs = os.listdir(item_path)
                for subdir in subdirs[:5]:
                    print(f"      └─ {subdir}")
                if len(subdirs) > 5:
                    print(f"      └─ ... and {len(subdirs) - 5} more")
            except:
                pass
        else:
            print(f"  📄 {item}")
else:
    print(f"Directory {kaggle_input} does not exist (not running in Kaggle)")


Using device: cuda
GPU: Tesla T4
GPU Memory: 15.64 GB

KAGGLE INPUT DIRECTORIES

Contents of /kaggle/input:
  📁 datasets/
      └─ raseluddin
  📁 models/
      └─ diptopodder
      └─ serhiikharchuk


## 2. Configuration


In [363]:
MODEL_PATH = "/kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1"
MODEL_NAME = MODEL_PATH
USE_UNSLOTH = True

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "lora_dropout": 0.1,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM
}

TRAINING_CONFIG = {
    "output_dir": "./results",
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "warmup_steps": 100,
    "logging_steps": 10,
    "eval_steps": 500,
    "save_steps": 500,
    "save_total_limit": 3,
    "fp16": False,
    "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    "gradient_checkpointing": True,
    "optim": "adamw_torch",
    "lr_scheduler_type": "cosine",
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False
}

DATASET_CONFIG = {
    "max_length": 1500,
    "do_not_reduce_sequence": True,
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1
}

print("Configuration loaded successfully!")


Configuration loaded successfully!


In [364]:
class DatasetProcessor:
    """
    Processes Bengali Empathetic Conversations dataset for LLM fine-tuning.
    """
    
    def __init__(self, tokenizer, max_length: int = 1500):
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def preprocess_text(self, text: str) -> str:
        """Preprocess Bengali text (normalization, cleaning)."""
        if text is None:
            return ""
        text = str(text)
        text = text.strip()
        return text
    
    def format_for_instruction(self, question: str, answer: str) -> str:
        """Format conversation for instruction tuning."""
        formatted = f"""### Instruction:
You are a helpful and empathetic Bengali assistant.

{question}

{answer}"""
        return formatted
    
    def tokenize_function(self, examples: Dict) -> Dict:
        """Tokenize examples with full sequence length preservation."""
        texts = []
        
        questions = examples.get('Questions', examples.get('input', []))
        answers = examples.get('Answers', examples.get('output', []))
        
        if not isinstance(questions, list):
            questions = [questions]
        if not isinstance(answers, list):
            answers = [answers]
        
        for i in range(len(questions)):
            question = questions[i] if i < len(questions) else ""
            answer = answers[i] if i < len(answers) else ""
            
            question = self.preprocess_text(question)
            answer = self.preprocess_text(answer)
            
            formatted_text = self.format_for_instruction(question, answer)
            texts.append(formatted_text)
        
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None
        )
        
        tokenized["labels"] = tokenized["input_ids"].copy()
        
        return tokenized
    
    def process_dataset(self, dataset, split: str = "train") -> HFDataset:
        """Process and tokenize dataset."""
        print(f"Processing {split} dataset...")
        
        tokenized_dataset = dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=dataset.column_names,
            desc=f"Tokenizing {split}"
        )
        
        print(f"{split} dataset processed. Size: {len(tokenized_dataset)}")
        return tokenized_dataset

print("DatasetProcessor class defined!")


DatasetProcessor class defined!


In [365]:
print("Loading dataset from Kaggle input...")
full_dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/*.csv"
)

full_dataset = full_dataset["train"]

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"Number of rows: {len(full_dataset)}")
print(f"Column names: {full_dataset.column_names}")
print(f"Features: {full_dataset.features}")

print("\n" + "=" * 60)
print("SAMPLE DATA (First 3 rows)")
print("=" * 60)
for i in range(min(3, len(full_dataset))):
    example = full_dataset[i]
    print(f"\nExample {i+1}:")
    for key, value in example.items():
        print(f"  {key}: {str(value)[:100]}...")

print("\n" + "=" * 60)
print("CLEANING DATASET (removing None/empty values)")
print("=" * 60)

def filter_empty(example):
    """Filter out examples with None or empty Questions/Answers."""
    question = example.get('Questions', '')
    answer = example.get('Answers', '')
    return (
        question is not None and 
        answer is not None and 
        str(question).strip() != '' and 
        str(answer).strip() != ''
    )

original_size = len(full_dataset)
full_dataset = full_dataset.filter(filter_empty)
cleaned_size = len(full_dataset)
removed = original_size - cleaned_size

print(f"Original size: {original_size}")
print(f"After cleaning: {cleaned_size}")
print(f"Removed: {removed} rows ({100*removed/original_size:.2f}%)")

dataset = full_dataset.train_test_split(test_size=DATASET_CONFIG["test_split"] + DATASET_CONFIG["val_split"], seed=42)
val_test = dataset["test"].train_test_split(test_size=DATASET_CONFIG["test_split"] / (DATASET_CONFIG["test_split"] + DATASET_CONFIG["val_split"]), seed=42)

train_dataset = dataset["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"\nDataset splits:")
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")


Loading dataset from Kaggle input...
DATASET INFORMATION
Number of rows: 38233
Column names: ['Topics', 'Question-Title', 'Questions', 'Answers']
Features: {'Topics': Value('string'), 'Question-Title': Value('string'), 'Questions': Value('string'), 'Answers': Value('string')}

SAMPLE DATA (First 3 rows)

Example 1:
  Topics: পারিবারিক দ্বন্দ্ব...
  Question-Title: মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি...
  Questions:  আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চলছে। অতীতে, তাদের মধ্যে ছোটখাটো পার্থক্য ছিল। উদাহরণস্...
  Answers:  আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা "ত্রিভুজকরণ" বলে অভিহিত করেছেন। যা হয় যখন পরিবারের একজন সদ...

Example 2:
  Topics: পদার্থের অপব্যবহার, আসক্তি...
  Question-Title: আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?...
  Questions:  আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ধূমপান ছেড়ে দিতে হবে - কিন্তু এটা কঠিন। কখনও কখনও এটি...
  Answers:  হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্যকর তা করার জন্য আগে থেকে পরিকল্পনা করা আপনার জন্য ভাল...

Example 3:
  T

## 5. Initialize Tokenizer and Model


In [366]:
print("Loading tokenizer from Kaggle model...")
print(f"Model path: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded successfully!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")
print("\nNote: If using Unsloth, tokenizer will be reloaded with optimizations in the next cell.")


Loading tokenizer from Kaggle model...
Model path: /kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1
Tokenizer loaded successfully!
Vocab size: 128000
Model max length: 131072
Pad token: <|end_of_text|>
EOS token: <|eot_id|>

Note: If using Unsloth, tokenizer will be reloaded with optimizations in the next cell.


In [367]:
processor = DatasetProcessor(tokenizer, max_length=DATASET_CONFIG["max_length"])
print("DatasetProcessor initialized!")


DatasetProcessor initialized!


In [ ]:
RESUME_FROM_SAVED = True
SAVED_ADAPTER_PATH = "/kaggle/input/models/diptopodder/checkpoint-1500/pytorch/default/1"

if USE_UNSLOTH:
    print("Loading model with Unsloth from Kaggle input...")
    
    import torch
    import gc
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU memory cleared. Available GPUs: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}, Memory: {props.total_memory / 1e9:.2f} GB")
    
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_PATH,
            max_seq_length=DATASET_CONFIG["max_length"],
            dtype=None,
            load_in_4bit=True,
            trust_remote_code=True
        )
        print(f"✓ Model loaded with max_seq_length={DATASET_CONFIG['max_length']}")
    except (ValueError, RuntimeError) as e:
        error_str = str(e)
        if "CPU or the disk" in error_str or "dispatched on the CPU" in error_str:
            print("⚠️ GPU memory issue detected. Reducing max_seq_length...")
            for reduced_length in [1024, 512]:
                try:
                    print(f"Trying with max_seq_length={reduced_length}...")
                    model, tokenizer = FastLanguageModel.from_pretrained(
                        model_name=MODEL_PATH,
                        max_seq_length=reduced_length,
                        dtype=None,
                        load_in_4bit=True,
                        trust_remote_code=True
                    )
                    print(f"✓ Model loaded with max_seq_length={reduced_length}")
                    print(f"⚠️ WARNING: max_seq_length reduced from {DATASET_CONFIG['max_length']} to {reduced_length}")
                    print("⚠️ Sequences will be truncated to this length during training.")
                    DATASET_CONFIG["max_length"] = reduced_length
                    break
                except Exception as e2:
                    if reduced_length == 512:
                        raise ValueError(f"Could not load model even with max_seq_length=512. Error: {e2}")
                    continue
        else:
            raise
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    tokenizer.padding_side = "right"
    
    print(f"\nTokenizer configuration:")
    print(f"  Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
    print(f"  EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
    print(f"  Padding side: {tokenizer.padding_side}")
    
    if RESUME_FROM_SAVED and os.path.exists(SAVED_ADAPTER_PATH):
        print("\n" + "="*60)
        print("⚠️ RESUMING TRAINING - Skipping LoRA adapter initialization")
        print("="*60)
        print("Base model loaded without LoRA adapters.")
        print("Saved adapter will be loaded in cell 11.5.")
        print("="*60 + "\n")
    else:
        print("\nAdding LoRA adapters for fresh training...")
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_CONFIG["r"],
            target_modules=LORA_CONFIG["target_modules"],
            lora_alpha=LORA_CONFIG["lora_alpha"],
            lora_dropout=LORA_CONFIG["lora_dropout"],
            bias="none",
            use_gradient_checkpointing=True,
            random_state=3407
        )
        print("Model loaded with Unsloth and LoRA!")
    
else:
    print("Loading model with standard PEFT LoRA from Kaggle input...")
    
    from transformers import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True
        )
    except ValueError as e:
        if "CPU or the disk" in str(e) or "dispatched on the CPU" in str(e):
            print("⚠️ GPU memory issue. Trying with CPU offload enabled...")
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                llm_int8_enable_fp32_cpu_offload=True
            )
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_PATH,
                torch_dtype=torch.bfloat16,
                device_map="auto",
                quantization_config=quantization_config,
                trust_remote_code=True
            )
            print("✓ Model loaded with CPU offload enabled")
        else:
            raise
    
    lora_config = LoraConfig(
        r=LORA_CONFIG["r"],
        lora_alpha=LORA_CONFIG["lora_alpha"],
        target_modules=LORA_CONFIG["target_modules"],
        lora_dropout=LORA_CONFIG["lora_dropout"],
        bias=LORA_CONFIG["bias"],
        task_type=LORA_CONFIG["task_type"]
    )
    
    model = get_peft_model(model, lora_config)
    
    print("Model loaded with PEFT LoRA!")

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")

print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)
print(f"Model type: {type(model).__name__}")
if hasattr(model, 'config'):
    print(f"Model config: {model.config.name_or_path if hasattr(model.config, 'name_or_path') else 'N/A'}")
    print(f"Model dtype: {next(model.parameters()).dtype}")
    print(f"Device: {next(model.parameters()).device}")
    if hasattr(model.config, 'hidden_size'):
        print(f"Hidden size: {model.config.hidden_size}")
    if hasattr(model.config, 'num_hidden_layers'):
        print(f"Number of layers: {model.config.num_hidden_layers}")
    if hasattr(model.config, 'num_attention_heads'):
        print(f"Number of attention heads: {model.config.num_attention_heads}")
    if hasattr(model.config, 'max_position_embeddings'):
        print(f"Max position embeddings: {model.config.max_position_embeddings}")

processor = DatasetProcessor(tokenizer, max_length=DATASET_CONFIG["max_length"])
print(f"\nDatasetProcessor reinitialized with final tokenizer!")
print(f"Using max_length: {DATASET_CONFIG['max_length']}")


Loading model with Unsloth from Kaggle input...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

## 6. Process Datasets


In [ ]:
print("=" * 60)
print("PROCESSING DATASETS")
print("=" * 60)
print("Note: Processing with final tokenizer (after Unsloth reload if applicable)")

train_tokenized = processor.process_dataset(train_dataset, split="train")
val_tokenized   = processor.process_dataset(val_dataset, split="val")
test_tokenized  = processor.process_dataset(test_dataset, split="test")

print("\n" + "=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)
print(f"Train tokenized: {len(train_tokenized)} samples")
print(f"Val tokenized: {len(val_tokenized)} samples")
print(f"Test tokenized: {len(test_tokenized)} samples")

print("\nDataset kept in list format (data collator will convert to tensors)")


In [369]:
print("=" * 60)
print("INSPECTING PROCESSED DATA")
print("=" * 60)

print("\nExample from TRAIN dataset (first sample):")
sample = train_tokenized[0]
print(f"Keys: {sample.keys()}")
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"Labels length: {len(sample['labels'])}")

decoded_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"\nDecoded text (first 500 chars):")
print(decoded_text[:500] + "...")

max_samples_for_stats = 1500
train_sample_size = min(max_samples_for_stats, len(train_tokenized))
val_sample_size = min(max_samples_for_stats, len(val_tokenized))
test_sample_size = min(max_samples_for_stats, len(test_tokenized))

train_lengths = [len(train_tokenized[i]['input_ids']) for i in range(train_sample_size)]
val_lengths = [len(val_tokenized[i]['input_ids']) for i in range(val_sample_size)]
test_lengths = [len(test_tokenized[i]['input_ids']) for i in range(test_sample_size)]

print("\n" + "=" * 60)
print("TOKEN LENGTH STATISTICS")
print("=" * 60)
print(f"\nTrain set (sampled {train_sample_size} of {len(train_tokenized)}):")
print(f"  Min length: {min(train_lengths)}")
print(f"  Max length: {max(train_lengths)}")
print(f"  Mean length: {np.mean(train_lengths):.1f}")
print(f"  Median length: {np.median(train_lengths):.1f}")
truncated_count = sum(1 for l in train_lengths if l > DATASET_CONFIG['max_length'])
print(f"  Samples > {DATASET_CONFIG['max_length']}: {truncated_count} ({100*truncated_count/train_sample_size:.1f}%)")

print(f"\nVal set (sampled {val_sample_size} of {len(val_tokenized)}):")
print(f"  Min length: {min(val_lengths)}")
print(f"  Max length: {max(val_lengths)}")
print(f"  Mean length: {np.mean(val_lengths):.1f}")
print(f"  Median length: {np.median(val_lengths):.1f}")

print(f"\nTest set (sampled {test_sample_size} of {len(test_tokenized)}):")
print(f"  Min length: {min(test_lengths)}")
print(f"  Max length: {max(test_lengths)}")
print(f"  Mean length: {np.mean(test_lengths):.1f}")
print(f"  Median length: {np.median(test_lengths):.1f}")

print("\n" + "=" * 60)
print("MORE EXAMPLES (showing formatted text)")
print("=" * 60)
for i in range(min(2, len(train_tokenized))):
    sample = train_tokenized[i]
    decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    print(f"\nExample {i+1} (length: {len(sample['input_ids'])} tokens):")
    print(decoded[:300] + "..." if len(decoded) > 300 else decoded)


INSPECTING PROCESSED DATA

Example from TRAIN dataset (first sample):
Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs length: 230
Labels length: 230

Decoded text (first 500 chars):
<|begin_of_text|>### Instruction:
You are a helpful and empathetic Bengali assistant.

### Input:
আমি এখন পর্যন্ত সেরা কাজ করেছি...সম্পূর্ণ স্ব-নিযুক্ত, ভাল খাওয়া, মহান বন্ধুদের সঙ্গ...মনে হচ্ছে জীবন আমাকে একটু শিথিল করতে শুরু করেছে

### Response:
এটি খুব সুন্দর, কারণ এটি সাধারণত ঘটে না।...

TOKEN LENGTH STATISTICS

Train set (sampled 1500 of 30568):
  Min length: 50
  Max length: 1500
  Mean length: 230.3
  Median length: 176.5
  Samples > 1500: 0 (0.0%)

Val set (sampled 1500 of 3821):
  Min length: 26
  Max length: 1500
  Mean length: 227.5
  Median length: 172.0

Test set (sampled 1500 of 3821):
  Min length: 48
  Max length: 1500
  Mean length: 223.3
  Median length: 172.0

MORE EXAMPLES (showing formatted text)

Example 1 (length: 230 tokens):
### Instruction:
You are a helpful a

## 7. Setup Training Arguments


In [370]:
import os
from transformers import TrainingArguments
from transformers.trainer_utils import IntervalStrategy

os.makedirs(TRAINING_CONFIG["output_dir"], exist_ok=True)

training_config = TRAINING_CONFIG.copy()

training_config["eval_strategy"] = IntervalStrategy.STEPS
training_config["save_strategy"] = IntervalStrategy.STEPS
training_config["eval_steps"] = 500
training_config["save_steps"] = 500
training_config["load_best_model_at_end"] = True

if "evaluation_strategy" in training_config:
    del training_config["evaluation_strategy"]

training_args = TrainingArguments(
    **training_config
)

if 'RESUME_FROM_SAVED' in globals() and RESUME_FROM_SAVED:
    if 'SAVED_ADAPTER_PATH' in globals() and os.path.exists(SAVED_ADAPTER_PATH):
        trainer_state_path = os.path.join(SAVED_ADAPTER_PATH, "trainer_state.json")
        if not os.path.exists(trainer_state_path):
            checkpoint_dir = training_config["output_dir"]
            if os.path.exists(checkpoint_dir):
                import re
                checkpoint_pattern = re.compile(r'checkpoint-(\d+)')
                for item in os.listdir(checkpoint_dir):
                    match = checkpoint_pattern.match(item)
                    if match:
                        checkpoint_path = os.path.join(checkpoint_dir, item)
                        trainer_state_path = os.path.join(checkpoint_path, "trainer_state.json")
                        if os.path.exists(trainer_state_path):
                            break
        
        if os.path.exists(trainer_state_path):
            try:
                import json
                with open(trainer_state_path, 'r') as f:
                    saved_trainer_state = json.load(f)
                
                if 'train_batch_size' in saved_trainer_state and saved_trainer_state['train_batch_size'] is not None:
                    saved_batch_size = saved_trainer_state['train_batch_size']
                    current_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
                    
                    if saved_batch_size != current_batch_size:
                        print(f"\n⚠️ Batch size mismatch detected!")
                        print(f"  Saved checkpoint batch size: {saved_batch_size}")
                        print(f"  Current configured batch size: {current_batch_size}")
                        print(f"  Adjusting to match saved batch size...")
                        
                        per_device_saved = saved_batch_size // training_args.gradient_accumulation_steps
                        if per_device_saved < 1:
                            per_device_saved = 1
                        
                        target_per_device = per_device_saved
                        
                        current_n_gpu = training_args.n_gpu if hasattr(training_args, 'n_gpu') and training_args.n_gpu > 1 else 1
                        
                        training_args.per_device_train_batch_size = target_per_device
                        training_args.per_device_eval_batch_size = target_per_device
                        
                        batch_without_n_gpu = target_per_device * training_args.gradient_accumulation_steps
                        batch_with_n_gpu = batch_without_n_gpu * current_n_gpu
                        
                        print(f"  Checkpoint batch size breakdown:")
                        print(f"    Saved train_batch_size: {saved_batch_size} (= per_device * gradient_accumulation)")
                        print(f"    Per device: {target_per_device}, Gradient accumulation: {training_args.gradient_accumulation_steps}")
                        print(f"  ✓ Updated per_device_train_batch_size to {target_per_device}")
                        print(f"  ✓ Batch size for step calculation: {batch_without_n_gpu} (matches checkpoint)")
                        print(f"  ✓ Actual effective batch (with {current_n_gpu} GPU(s)): {batch_with_n_gpu}")
                        print(f"  Note: Step calculation uses batch_size={batch_without_n_gpu} to match checkpoint")
            except Exception as e:
                print(f"⚠️ Could not read saved batch size: {e}")
                print(f"  Using configured batch size: {training_args.per_device_train_batch_size}")

print("Training arguments configured successfully!")
print(f"Evaluation strategy: {training_args.eval_strategy}")
print(f"Save strategy: {training_args.save_strategy}")
print(f"Load best model at end: {training_args.load_best_model_at_end}")
print(f"Per device train batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")




Training arguments configured successfully!
Evaluation strategy: IntervalStrategy.STEPS
Save strategy: SaveStrategy.STEPS
Load best model at end: True
Per device train batch size: 1
Gradient accumulation steps: 8
Effective batch size: 8


## 8. Data Collator


In [371]:
import torch

class CustomDataCollator:
    """Custom data collator that manually handles padding for variable-length sequences."""
    
    def __init__(self, tokenizer, pad_to_multiple_of=None, max_length=None):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of
        self.max_length = max_length
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        
    def __call__(self, features):
        """Collate features into a batch with proper padding."""
        batch = {}
        
        input_ids = []
        labels = []
        attention_mask = []
        
        for f in features:
            ids = f['input_ids']
            if not isinstance(ids, list):
                ids = ids.tolist() if hasattr(ids, 'tolist') else list(ids)
            
            if self.max_length is not None and len(ids) > self.max_length:
                ids = ids[:self.max_length]
            
            input_ids.append(ids)
            
            lbls = f.get('labels', ids.copy())
            if not isinstance(lbls, list):
                lbls = lbls.tolist() if hasattr(lbls, 'tolist') else list(lbls)
            
            if len(lbls) > len(ids):
                lbls = lbls[:len(ids)]
            
            labels.append(lbls)
            
            attn = f.get('attention_mask', [1] * len(ids))
            if not isinstance(attn, list):
                attn = attn.tolist() if hasattr(attn, 'tolist') else list(attn)
            
            if len(attn) > len(ids):
                attn = attn[:len(ids)]
            
            attention_mask.append(attn)
        
        max_length = max(len(ids) for ids in input_ids)
        
        if self.max_length is not None:
            max_length = min(max_length, self.max_length)
        
        if self.pad_to_multiple_of:
            max_length = ((max_length + self.pad_to_multiple_of - 1) // self.pad_to_multiple_of) * self.pad_to_multiple_of
        
        padded_input_ids = []
        padded_labels = []
        padded_attention_mask = []
        
        for ids, lbls, attn in zip(input_ids, labels, attention_mask):
            pad_length = max_length - len(ids)
            
            padded_ids = list(ids) + [self.pad_token_id] * pad_length
            padded_input_ids.append(padded_ids)
            
            padded_lbls = list(lbls) + [-100] * pad_length
            padded_labels.append(padded_lbls)
            
            padded_attn = list(attn) + [0] * pad_length
            padded_attention_mask.append(padded_attn)
        
        batch['input_ids'] = torch.tensor(padded_input_ids, dtype=torch.long)
        batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
        batch['attention_mask'] = torch.tensor(padded_attention_mask, dtype=torch.long)
        
        return batch

data_collator = CustomDataCollator(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    max_length=DATASET_CONFIG["max_length"]
)

print(f"Tokenizer padding side: {tokenizer.padding_side}")
print(f"Tokenizer pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
if tokenizer.pad_token is None:
    print("⚠️ WARNING: Pad token is None! Setting it to EOS token.")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Data collator created!")
print(f"Pad token ID: {tokenizer.pad_token_id}")
print(f"EOS token ID: {tokenizer.eos_token_id}")

print("\nVerifying tokenized data format...")
sample = train_tokenized[0]
print(f"Sample keys: {sample.keys()}")
print(f"Input IDs type: {type(sample['input_ids'])}, length: {len(sample['input_ids'])}")
print(f"Labels type: {type(sample['labels'])}, length: {len(sample['labels'])}")
print(f"Attention mask type: {type(sample.get('attention_mask', 'N/A'))}")

if isinstance(sample['labels'], list):
    print(f"✓ Labels are in list format (correct)")
    if len(sample['labels']) > 0:
        print(f"  First label value: {sample['labels'][0]} (type: {type(sample['labels'][0])})")
else:
    print(f"⚠ Labels are not in list format: {type(sample['labels'])}")



Tokenizer padding side: right
Tokenizer pad token: <|end_of_text|> (ID: 128001)
Data collator created!
Pad token ID: 128001
EOS token ID: 128009

Verifying tokenized data format...
Sample keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input IDs type: <class 'list'>, length: 230
Labels type: <class 'list'>, length: 230
Attention mask type: <class 'list'>
✓ Labels are in list format (correct)
  First label value: 128000 (type: <class 'int'>)


## 9. Experiment Logging


In [357]:
class ExperimentLogger:
    """Logs experiments to LLAMAExperiments table structure."""
    
    def __init__(self, log_dir: str = "./logs"):
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        self.experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiment_log = {
            "id": self.experiment_id,
            "model_path": MODEL_PATH,
            "lora_config": LORA_CONFIG,
            "training_config": TRAINING_CONFIG,
            "train_loss": [],
            "val_loss": [],
            "metrics": {},
            "timestamp": datetime.now().isoformat()
        }
        
    def log_training_step(self, step: int, loss: float):
        """Log training step loss."""
        self.experiment_log["train_loss"].append({"step": step, "loss": loss})
        
    def log_validation(self, step: int, val_loss: float, metrics: Dict = None):
        """Log validation results."""
        self.experiment_log["val_loss"].append({"step": step, "loss": val_loss})
        if metrics:
            self.experiment_log["metrics"][step] = metrics
        
    def save(self):
        """Save experiment log to JSON."""
        log_path = os.path.join(self.log_dir, f"experiment_{self.experiment_id}.json")
        with open(log_path, 'w', encoding='utf-8') as f:
            json.dump(self.experiment_log, f, indent=2, ensure_ascii=False)
        print(f"Experiment log saved to {log_path}")
        return log_path

logger = ExperimentLogger()
print(f"Experiment ID: {logger.experiment_id}")


Experiment ID: 20260220_064649


## 10. Custom Trainer with Logging


In [358]:
class CustomTrainer(Trainer):
    """Custom trainer with experiment logging."""
    
    def __init__(self, experiment_logger, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.experiment_logger = experiment_logger
    
    def log(self, logs: Dict[str, float], *args, **kwargs):
        """Override log to capture training metrics.
        
        Args:
            logs: Dictionary of metrics to log
            *args: Additional positional arguments (like start_time) that may be passed by Trainer
            **kwargs: Additional keyword arguments that may be passed by Trainer
        """
        super().log(logs, *args, **kwargs)
        
        if "loss" in logs:
            step = logs.get("step", 0)
            self.experiment_logger.log_training_step(step, logs["loss"])
        
        if "eval_loss" in logs:
            step = logs.get("step", 0)
            metrics = {k: v for k, v in logs.items() if k not in ["step", "eval_loss"]}
            self.experiment_logger.log_validation(step, logs["eval_loss"], metrics)

print("CustomTrainer class defined!")


CustomTrainer class defined!


## 11. Initialize Trainer


In [ ]:
print("=" * 60)
print("PRE-TRAINER VERIFICATION")
print("=" * 60)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100*trainable_params/total_params:.2f}%")

has_peft = hasattr(model, 'peft_config') and len(model.peft_config) > 0
print(f"\nHas PEFT adapters: {has_peft}")

if has_peft:
    print(f"Number of PEFT adapters: {len(model.peft_config)}")
    for adapter_name, adapter_cfg in model.peft_config.items():
        print(f"  - {adapter_name}: r={adapter_cfg.r}, alpha={adapter_cfg.lora_alpha}, type={adapter_cfg.peft_type}")
    
    from peft import PeftModel
    is_peft_model = isinstance(model, PeftModel)
    print(f"Is PeftModel instance: {is_peft_model}")
    
    if not is_peft_model and hasattr(model, 'base_model'):
        print("⚠️ Model has adapters but is not a PeftModel instance")
        print("This might cause Unsloth to not recognize the adapters")
else:
    print("⚠️ WARNING: Model does not have PEFT adapters!")
    print("Unsloth requires PEFT adapters for training quantized models")

if trainable_params == 0:
    print("\n" + "=" * 60)
    print("❌ CRITICAL ERROR: Model has 0 trainable parameters!")
    print("=" * 60)
    print("The adapter was not loaded correctly or is not attached to the model.")
    print("\nPossible causes:")
    print("1. Adapter loading cell (11.5) was not run or failed silently")
    print("2. RESUME_FROM_SAVED was not set to True in Cell 15")
    print("3. Adapter files not found at the specified path")
    print("4. Model is in inference mode (should be in training mode)")
    
    print("\nModel diagnostics:")
    print(f"  - Model type: {type(model)}")
    print(f"  - Has peft_config: {hasattr(model, 'peft_config')}")
    if hasattr(model, 'peft_config'):
        print(f"  - Number of adapters: {len(model.peft_config)}")
        for name, config in model.peft_config.items():
            print(f"    - {name}: r={config.r}, alpha={config.lora_alpha}")
    print(f"  - Model training mode: {model.training if hasattr(model, 'training') else 'N/A'}")
    
    if 'RESUME_FROM_SAVED' in globals():
        print(f"\n  - RESUME_FROM_SAVED: {RESUME_FROM_SAVED}")
    else:
        print("\n  - RESUME_FROM_SAVED: NOT SET (should be True)")
    
    if 'SAVED_ADAPTER_PATH' in globals():
        print(f"  - SAVED_ADAPTER_PATH: {SAVED_ADAPTER_PATH}")
        print(f"  - Path exists: {os.path.exists(SAVED_ADAPTER_PATH)}")
        if os.path.exists(SAVED_ADAPTER_PATH):
            adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
            for f in adapter_files:
                path = os.path.join(SAVED_ADAPTER_PATH, f)
                print(f"    - {f}: {'✓' if os.path.exists(path) else '✗'}")
    
    print("\n" + "=" * 60)
    print("SOLUTION:")
    print("1. Go back to Cell 15 and set RESUME_FROM_SAVED = True")
    print("2. Re-run Cell 15 (model loading)")
    print("3. Re-run Cell 11.5 (adapter loading)")
    print("4. Verify trainable parameters > 0")
    print("5. Then run this cell again")
    print("=" * 60)
    raise ValueError("Cannot initialize trainer: Model has 0 trainable parameters!")

if trainable_params > 0 and not has_peft:
    print("\n" + "=" * 60)
    print("⚠️ WARNING: Trainable parameters exist but PEFT adapters not detected!")
    print("=" * 60)
    print("The model was likely loaded with LoRA in Cell 15 instead of as a base model.")
    print("This happens when RESUME_FROM_SAVED was not True when Cell 15 ran.")
    print("\nAttempting to fix by reloading the adapter properly...")
    
    if 'SAVED_ADAPTER_PATH' not in globals():
        SAVED_ADAPTER_PATH = "/kaggle/input/models/diptopodder/checkpoint-1500/pytorch/default/1"
    
    if os.path.exists(SAVED_ADAPTER_PATH):
        print(f"\nFound saved adapter at: {SAVED_ADAPTER_PATH}")
        print("Reloading model with adapter...")
        
        try:
            from peft import PeftModel
            
            print("Extracting base model...")
            
            if hasattr(model, 'get_base_model'):
                base_model = model.get_base_model()
                if not hasattr(base_model, 'prepare_inputs_for_generation'):
                    if hasattr(model, 'base_model'):
                        base_model = model.base_model
            elif hasattr(model, 'base_model'):
                base_model = model.base_model
                if not hasattr(base_model, 'prepare_inputs_for_generation'):
                    print("⚠️ Extracted model is not full model. Reloading...")
                    base_model = None
            else:
                base_model = None
            
            if base_model is None or not hasattr(base_model, 'prepare_inputs_for_generation'):
                print("Reloading base model fresh (this ensures we have the full model)...")
                if USE_UNSLOTH:
                    base_model, tokenizer = FastLanguageModel.from_pretrained(
                        model_name=MODEL_PATH,
                        max_seq_length=DATASET_CONFIG["max_length"],
                        dtype=None,
                        load_in_4bit=True,
                        trust_remote_code=True
                    )
                else:
                    base_model = AutoModelForCausalLM.from_pretrained(
                        MODEL_PATH,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                        load_in_4bit=True,
                        trust_remote_code=True
                    )
            
            if not hasattr(base_model, 'prepare_inputs_for_generation'):
                raise ValueError(
                    "❌ Extracted model is not a full LlamaForCausalLM!\n"
                    "It's missing 'prepare_inputs_for_generation' method.\n"
                    "Please re-run Cell 15 with RESUME_FROM_SAVED = True to load base model correctly."
                )
            
            print("Loading adapter onto base model (is_trainable=True)...")
            model = PeftModel.from_pretrained(base_model, SAVED_ADAPTER_PATH, is_trainable=True)
            
            try:
                if hasattr(model, 'enable_adapter_layers'):
                    model.enable_adapter_layers()
                    print("✓ Enabled adapter layers for training")
            except:
                pass
            model.train()
            
            has_peft_after = hasattr(model, 'peft_config') and len(model.peft_config) > 0
            trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
            total_after = sum(p.numel() for p in model.parameters())
            
            print(f"\n✓ Adapter reloaded!")
            print(f"  - Has PEFT adapters: {has_peft_after}")
            print(f"  - Trainable parameters: {trainable_after:,} / {total_after:,} ({100*trainable_after/total_after:.2f}%)")
            
            if has_peft_after:
                print("✓ PEFT adapters now properly registered!")
                has_peft = True
                trainable_params = trainable_after
                total_params = total_after
            else:
                raise ValueError("Adapter reloaded but PEFT config still missing!")
                
        except Exception as e:
            print(f"\n❌ Failed to reload adapter: {e}")
            print("\nPlease manually:")
            print("1. Go to Cell 15 and set RESUME_FROM_SAVED = True")
            print("2. Re-run Cell 15 (this will load base model without LoRA)")
            print("3. Re-run Cell 29 (this will load the saved adapter)")
            print("4. Then run this cell again")
            raise ValueError(f"Could not automatically fix: {e}")
    else:
        print(f"\n❌ Saved adapter path not found: {SAVED_ADAPTER_PATH}")
        print("Please check the path and try again.")
        raise ValueError("Cannot find saved adapter to reload")
    
    if not hasattr(model, 'peft_config') or len(model.peft_config) == 0:
        raise ValueError(
            "❌ CRITICAL: Model has trainable parameters but no PEFT adapters!\n"
            "Unsloth Trainer requires PEFT adapters to be properly registered.\n"
            "Please re-run Cell 11.5 (adapter loading) to ensure adapter is loaded correctly."
        )

if not has_peft:
    raise ValueError(
        "❌ CRITICAL: Model does not have PEFT adapters!\n"
        "Unsloth Trainer requires PEFT adapters for training quantized models.\n"
        "Please ensure Cell 11.5 (adapter loading) was run successfully."
    )

print("\n✓ Model verification passed!")
print("✓ Trainable parameters: OK")
print("✓ PEFT adapters: OK")
print("=" * 60)

print("\n" + "=" * 60)
print("FINAL MODEL CHECK BEFORE TRAINING")
print("=" * 60)

model.train()
print(f"✓ Model training mode: {model.training}")

trainable_final = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Trainable parameters: {trainable_final:,}")

if hasattr(model, 'peft_config'):
    print(f"✓ PEFT adapters: {len(model.peft_config)}")
    for adapter_name, adapter_cfg in model.peft_config.items():
        if hasattr(adapter_cfg, 'inference_mode'):
            print(f"  - {adapter_name}: inference_mode={adapter_cfg.inference_mode}")
else:
    print("⚠️ WARNING: No PEFT config found!")

lora_params_trainable = sum(
    p.numel() for name, p in model.named_parameters() 
    if 'lora' in name.lower() and p.requires_grad
)
print(f"✓ LoRA parameters trainable: {lora_params_trainable:,}")

if lora_params_trainable == 0:
    print("\n⚠️ WARNING: No LoRA parameters are trainable!")
    print("Attempting to enable training on all LoRA parameters...")
    for name, param in model.named_parameters():
        if 'lora' in name.lower():
            param.requires_grad = True
    lora_params_trainable = sum(
        p.numel() for name, p in model.named_parameters() 
        if 'lora' in name.lower() and p.requires_grad
    )
    print(f"After fix: {lora_params_trainable:,} LoRA parameters trainable")

print("=" * 60)

try:
    from accelerate.state import AcceleratorState
    from accelerate import PartialState
    
    try:
        state = PartialState()
        print(f"✓ AcceleratorState initialized: {state.distributed_type}")
    except Exception as init_error:
        try:
            if hasattr(AcceleratorState, '_reset_state'):
                AcceleratorState._reset_state()
            state = PartialState()
            print(f"✓ AcceleratorState reinitialized: {state.distributed_type}")
        except Exception as reset_error:
            print(f"⚠️ Could not initialize AcceleratorState: {reset_error}")
            print("Trainer will attempt to initialize it")
except Exception as e:
    print(f"⚠️ Note: {e}")
    print("Trainer will initialize accelerator state")

trainer = CustomTrainer(
    experiment_logger=logger,
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Trainer initialized!")

model.train()
print(f"✓ Model training mode after trainer init: {model.training}")

print("\nEnsuring all LoRA parameters have requires_grad=True...")
lora_param_count = 0
for name, param in model.named_parameters():
    if 'lora' in name.lower() or 'adapter' in name.lower():
        if not param.requires_grad:
            param.requires_grad = True
            lora_param_count += 1

if lora_param_count > 0:
    print(f"✓ Enabled training on {lora_param_count} additional LoRA parameter groups")

final_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Final trainable parameters: {final_trainable:,}")

if USE_UNSLOTH and hasattr(model, 'gradient_checkpointing_enable'):
    try:
        model.gradient_checkpointing_enable()
        print("✓ Gradient checkpointing enabled")
    except:
        pass

model.train()
print(f"✓ Final model training mode: {model.training}")


PRE-TRAINER VERIFICATION
Trainable parameters: 525,602,816
Total parameters: 4,540,600,320
Trainable percentage: 11.58%

Has PEFT adapters: False
⚠️ WARNING: Model does not have PEFT adapters!
Unsloth requires PEFT adapters for training quantized models

⚠️ WARNING: Trainable parameters exist but PEFT adapters not detected!
The model was likely loaded with LoRA in Cell 15 instead of as a base model.
This happens when RESUME_FROM_SAVED was not True when Cell 15 ran.

Attempting to fix by reloading the adapter properly...

Found saved adapter at: /kaggle/input/models/diptopodder/checkpoint-1500/pytorch/default/1
Reloading model with adapter...
Extracting base model...
⚠️ Extracted model is not full model. Reloading...
Reloading base model fresh (this ensures we have the full model)...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPU

ValueError: Could not automatically fix: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [297]:

if 'RESUME_FROM_SAVED' not in globals():
    RESUME_FROM_SAVED = True
if 'SAVED_ADAPTER_PATH' not in globals():
    SAVED_ADAPTER_PATH = "/kaggle/input/models/diptopodder/checkpoint-1500/pytorch/default/1"


if RESUME_FROM_SAVED and os.path.exists(SAVED_ADAPTER_PATH):
    print("=" * 60)
    print("LOADING SAVED ADAPTER")
    print("=" * 60)
    print(f"Loading from: {SAVED_ADAPTER_PATH}")
    
    adapter_config_path = os.path.join(SAVED_ADAPTER_PATH, "adapter_config.json")
    adapter_model_path = os.path.join(SAVED_ADAPTER_PATH, "adapter_model.safetensors")
    
    if os.path.exists(adapter_config_path) and os.path.exists(adapter_model_path):
        print("Found saved adapter files!")
        
        if 'model' not in locals() or model is None:
            print("Loading base model...")
            if USE_UNSLOTH:
                model, tokenizer = FastLanguageModel.from_pretrained(
                    model_name=MODEL_PATH,
                    max_seq_length=DATASET_CONFIG["max_length"],
                    dtype=None,
                    load_in_4bit=True,
                    trust_remote_code=True
                )
            else:
                model = AutoModelForCausalLM.from_pretrained(
                    MODEL_PATH,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    load_in_4bit=True,
                    trust_remote_code=True
                )
        
        print("Loading saved LoRA adapter...")
        from peft import PeftModel, PeftConfig
        
        try:
            adapter_config = PeftConfig.from_pretrained(SAVED_ADAPTER_PATH)
            print(f"✓ Adapter config verified: r={adapter_config.r}, alpha={adapter_config.lora_alpha}")
        except Exception as e:
            print(f"⚠️ Warning: Could not load adapter config: {e}")
        
        if USE_UNSLOTH:
            print("Loading adapter with Unsloth compatibility...")
            
            has_existing_adapters = hasattr(model, 'peft_config') and len(model.peft_config) > 0
            
            if has_existing_adapters:
                print("⚠️ Model has existing adapters. Extracting base model...")
                if hasattr(model, 'get_base_model'):
                    base_model = model.get_base_model()
                elif hasattr(model, 'base_model'):
                    if hasattr(model.base_model, 'model'):
                        base_model = model.base_model.model
                    else:
                        base_model = model.base_model
                else:
                    print("Cannot extract base model. Reloading...")
                    base_model, tokenizer = FastLanguageModel.from_pretrained(
                        model_name=MODEL_PATH,
                        max_seq_length=DATASET_CONFIG["max_length"],
                        dtype=None,
                        load_in_4bit=True,
                        trust_remote_code=True
                    )
                
                print("Loading adapter using Unsloth-compatible method...")
                
                adapter_config = PeftConfig.from_pretrained(SAVED_ADAPTER_PATH)
                print(f"✓ Adapter config loaded: r={adapter_config.r}, alpha={adapter_config.lora_alpha}")
                
                print("Creating LoRA adapter structure with Unsloth...")
                model = FastLanguageModel.get_peft_model(
                    base_model,
                    r=adapter_config.r,
                    target_modules=adapter_config.target_modules,
                    lora_alpha=adapter_config.lora_alpha,
                    lora_dropout=adapter_config.lora_dropout,
                    bias=adapter_config.bias if hasattr(adapter_config, 'bias') else "none",
                    use_gradient_checkpointing=True,
                    random_state=3407
                )
                print("✓ Adapter structure created")
                
                print("Loading saved adapter weights...")
                from safetensors.torch import load_file
                from peft import get_peft_model_state_dict
                
                adapter_weights_path = os.path.join(SAVED_ADAPTER_PATH, "adapter_model.safetensors")
                if not os.path.exists(adapter_weights_path):
                    raise FileNotFoundError(f"Adapter weights not found: {adapter_weights_path}")
                
                adapter_weights = load_file(adapter_weights_path)
                current_state = get_peft_model_state_dict(model)
                
                loaded_count = 0
                for key, param in current_state.items():
                    if key in adapter_weights:
                        param.data.copy_(adapter_weights[key])
                        loaded_count += 1
                
                print(f"✓ Loaded {loaded_count}/{len(current_state)} weight tensors")
                
                if hasattr(model, 'peft_config'):
                    for adapter_name, adapter_cfg in model.peft_config.items():
                        if hasattr(adapter_cfg, 'inference_mode') and adapter_cfg.inference_mode:
                            adapter_cfg.inference_mode = False
                            print(f"✓ Disabled inference_mode for adapter '{adapter_name}'")
            else:
                print("Model has no existing adapters. Loading saved adapter...")
                
                print("Loading adapter using Unsloth-compatible method...")
                
                adapter_config = PeftConfig.from_pretrained(SAVED_ADAPTER_PATH)
                print(f"✓ Adapter config loaded: r={adapter_config.r}, alpha={adapter_config.lora_alpha}")
                
                print("Creating LoRA adapter structure with Unsloth...")
                model = FastLanguageModel.get_peft_model(
                    model,
                    r=adapter_config.r,
                    target_modules=adapter_config.target_modules,
                    lora_alpha=adapter_config.lora_alpha,
                    lora_dropout=adapter_config.lora_dropout,
                    bias=adapter_config.bias if hasattr(adapter_config, 'bias') else "none",
                    use_gradient_checkpointing=True,
                    random_state=3407
                )
                print("✓ Adapter structure created with Unsloth")
                
                print("Loading saved adapter weights...")
                from safetensors.torch import load_file
                from peft import get_peft_model_state_dict
                
                adapter_weights_path = os.path.join(SAVED_ADAPTER_PATH, "adapter_model.safetensors")
                if not os.path.exists(adapter_weights_path):
                    raise FileNotFoundError(f"Adapter weights not found: {adapter_weights_path}")
                
                adapter_weights = load_file(adapter_weights_path)
                current_state = get_peft_model_state_dict(model)
                
                loaded_count = 0
                for key, param in current_state.items():
                    if key in adapter_weights:
                        param.data.copy_(adapter_weights[key])
                        loaded_count += 1
                
                print(f"✓ Loaded {loaded_count}/{len(current_state)} weight tensors")
                
                if hasattr(model, 'peft_config'):
                    for adapter_name, adapter_cfg in model.peft_config.items():
                        if hasattr(adapter_cfg, 'inference_mode') and adapter_cfg.inference_mode:
                            adapter_cfg.inference_mode = False
                            print(f"✓ Disabled inference_mode for adapter '{adapter_name}'")
                
                model.train()
                print("✓ Model set to training mode")
            
            print("\n" + "=" * 60)
            print("VERIFYING ADAPTER LOADING")
            print("=" * 60)
            
            from peft import PeftModel
            is_peft = isinstance(model, PeftModel)
            has_peft_config = hasattr(model, 'peft_config') and len(model.peft_config) > 0
            
            print(f"Is PeftModel: {is_peft}")
            print(f"Has peft_config: {has_peft_config}")
            
            if has_peft_config:
                print(f"Number of adapters: {len(model.peft_config)}")
                for adapter_name, adapter_cfg in model.peft_config.items():
                    print(f"  - Adapter '{adapter_name}': r={adapter_cfg.r}, alpha={adapter_cfg.lora_alpha}, type={adapter_cfg.peft_type}")
                    
                    if hasattr(adapter_cfg, 'inference_mode') and adapter_cfg.inference_mode:
                        print(f"    ⚠️ inference_mode=True detected. Disabling...")
                        adapter_cfg.inference_mode = False
                        
                        for name, param in model.named_parameters():
                            if 'lora' in name.lower() or adapter_name in name.lower():
                                param.requires_grad = True
                        print(f"    ✓ Disabled inference mode and enabled training")
            else:
                raise ValueError(
                    "❌ CRITICAL: Adapter loaded but peft_config is missing!\n"
                    "This will cause Unsloth Trainer to fail.\n"
                    "The adapter was not properly registered as a PEFT model."
                )
            
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            total = sum(p.numel() for p in model.parameters())
            trainable_percent = 100 * trainable / total if total > 0 else 0
            
            print(f"\nTrainable parameters: {trainable:,} / {total:,} ({trainable_percent:.2f}%)")
            
            if trainable == 0:
                print("⚠️ Trainable parameters = 0. Attempting to enable training on adapter parameters...")
                
                adapter_param_count = 0
                for name, param in model.named_parameters():
                    if any(keyword in name.lower() for keyword in ['lora', 'adapter', 'peft']):
                        param.requires_grad = True
                        adapter_param_count += 1
                
                print(f"Enabled training on {adapter_param_count} adapter parameter groups")
                
                trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                trainable_percent = 100 * trainable / total if total > 0 else 0
                print(f"Trainable parameters after fix: {trainable:,} / {total:,} ({trainable_percent:.2f}%)")
                
                if trainable == 0:
                    print("⚠️ Still 0 trainable params. Trying alternative method...")
                    
                    try:
                        if hasattr(model, 'enable_adapter_layers'):
                            model.enable_adapter_layers()
                            print("✓ Enabled adapter layers using enable_adapter_layers()")
                    except:
                        pass
                    
                    for name, module in model.named_modules():
                        if 'lora' in name.lower() or 'adapter' in name.lower():
                            for param in module.parameters():
                                param.requires_grad = True
                    
                    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                    print(f"Trainable parameters after method 2: {trainable:,}")
                    
                    if trainable == 0:
                        raise ValueError(
                            "❌ CRITICAL ERROR: Trainable parameters = 0 after all fix attempts!\n"
                            "This means the adapter was not loaded correctly.\n"
                            "Possible solutions:\n"
                            "1. Make sure RESUME_FROM_SAVED = True in Cell 15 (before model loading)\n"
                            "2. Check that adapter files exist at the path\n"
                            "3. Verify the adapter was saved correctly\n"
                            "4. The adapter might not be compatible with the quantized model structure\n"
                            "5. Try reloading the base model and adapter from scratch"
                        )
            
            if hasattr(model, 'training'):
                if not model.training:
                    print("⚠️ Model not in training mode. Setting to training mode...")
                    model.train()
                print(f"✓ Model training mode: {model.training}")
            else:
                print("⚠️ WARNING: Cannot verify model training mode")
            
            print("=" * 60)
            print("✓ Adapter loaded successfully with PEFT adapters registered!")
            print("=" * 60)
            
            if USE_UNSLOTH:
                print("\nApplying Unsloth-specific training setup...")
                
                try:
                    if hasattr(model, 'enable_adapter_layers'):
                        model.enable_adapter_layers()
                        print("✓ Enabled adapter layers using enable_adapter_layers()")
                except Exception as e:
                    print(f"⚠️ Could not use enable_adapter_layers(): {e}")
                
                print("Ensuring all LoRA parameters are trainable...")
                lora_enabled = 0
                for name, param in model.named_parameters():
                    if 'lora' in name.lower():
                        if not param.requires_grad:
                            param.requires_grad = True
                            lora_enabled += 1
                
                if lora_enabled > 0:
                    print(f"✓ Enabled training on {lora_enabled} LoRA parameter groups")
                
                if hasattr(model, 'peft_config'):
                    for adapter_name, adapter_cfg in model.peft_config.items():
                        if hasattr(adapter_cfg, 'inference_mode') and adapter_cfg.inference_mode:
                            adapter_cfg.inference_mode = False
                            print(f"✓ Disabled inference_mode for adapter '{adapter_name}'")
                
                model.train()
                
                final_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                print(f"✓ Final trainable parameters: {final_trainable:,}")
                print(f"✓ Model training mode: {model.training}")
                
                try:
                    if not isinstance(model, FastLanguageModel):
                        print("Ensuring Unsloth compatibility...")
                        
                        if not hasattr(model, 'prepare_inputs_for_generation'):
                            print("⚠️ Model missing prepare_inputs_for_generation - this might cause issues")
                        
                        if hasattr(model, 'gradient_checkpointing_enable'):
                            try:
                                model.gradient_checkpointing_enable()
                                print("✓ Gradient checkpointing enabled")
                            except:
                                pass
                        
                except Exception as e:
                    print(f"⚠️ Could not apply Unsloth compatibility fixes: {e}")
                
                print("✓ Unsloth training setup complete!")
                
        else:
            if hasattr(model, 'peft_config') and len(model.peft_config) > 0:
                if hasattr(model, 'get_base_model'):
                    base_model = model.get_base_model()
                elif hasattr(model, 'base_model'):
                    base_model = model.base_model.model if hasattr(model.base_model, 'model') else model.base_model
                else:
                    base_model = model
                model = PeftModel.from_pretrained(base_model, SAVED_ADAPTER_PATH, is_trainable=True)
            else:
                model = PeftModel.from_pretrained(model, SAVED_ADAPTER_PATH, is_trainable=True)
        
        print("✓ Saved adapter loaded successfully!")
        
        optimizer_path = os.path.join(SAVED_ADAPTER_PATH, "optimizer.pt")
        scheduler_path = os.path.join(SAVED_ADAPTER_PATH, "scheduler.pt")
        training_args_path = os.path.join(SAVED_ADAPTER_PATH, "training_args.bin")
        
        saved_training_state = {}
        if os.path.exists(optimizer_path):
            print("Found optimizer state (will be loaded by trainer if resuming)")
            saved_training_state['optimizer'] = optimizer_path
        if os.path.exists(scheduler_path):
            print("Found scheduler state (will be loaded by trainer if resuming)")
            saved_training_state['scheduler'] = scheduler_path
        if os.path.exists(training_args_path):
            print("Found training args (will be loaded by trainer if resuming)")
            saved_training_state['training_args'] = training_args_path
        
        ESTIMATED_STEP = 1000
        print(f"\nNote: Estimated training step: {ESTIMATED_STEP}")
        print("You may need to manually adjust this based on when you saved the model.")
        print("=" * 60)
        
    else:
        print(f"⚠️ Adapter files not found in {SAVED_ADAPTER_PATH}")
        print("Continuing with fresh model...")
else:
    print("Not loading saved adapter - starting fresh training")


LOADING SAVED ADAPTER
Loading from: /kaggle/input/models/diptopodder/checkpoint-1500/pytorch/default/1
Found saved adapter files!
Loading saved LoRA adapter...
✓ Adapter config verified: r=16, alpha=32
Loading adapter with Unsloth compatibility...
⚠️ Model has existing adapters. Extracting base model...
Loading adapter using Unsloth-compatible method...
✓ Adapter config loaded: r=16, alpha=32
Creating LoRA adapter structure with Unsloth...
✓ Adapter structure created
Loading saved adapter weights...
✓ Loaded 256/256 weight tensors

VERIFYING ADAPTER LOADING
Is PeftModel: True
Has peft_config: True
Number of adapters: 1
  - Adapter 'default': r=16, alpha=32, type=PeftType.LORA

Trainable parameters: 13,631,488 / 4,554,231,808 (0.30%)
✓ Model training mode: True
✓ Adapter loaded successfully with PEFT adapters registered!

Applying Unsloth-specific training setup...
✓ Enabled adapter layers using enable_adapter_layers()
Ensuring all LoRA parameters are trainable...
✓ Final trainable para

In [298]:

print("=" * 60)
print("VERIFYING ADAPTER LOADING")
print("=" * 60)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / total_params if total_params > 0 else 0

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {trainable_percent:.2f}%")

if trainable_params == 0:
    print("\n❌ ERROR: Trainable parameters = 0!")
    print("This means the adapter was NOT loaded correctly.")
    print("\nPossible causes:")
    print("1. Adapter files not found at the specified path")
    print("2. Model was put in inference mode")
    print("3. Adapter loading failed silently")
    print("\nTrying to fix by ensuring model is in training mode...")
    
    model.train()
    
    if hasattr(model, 'peft_config'):
        print(f"PEFT config found: {len(model.peft_config)} adapter(s)")
        if len(model.peft_config) == 0:
            print("⚠️ WARNING: No PEFT adapters found! You may need to reload the adapter.")
    else:
        print("⚠️ WARNING: Model does not have peft_config attribute!")
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nAfter fix attempt - Trainable parameters: {trainable_params:,}")
    
    if trainable_params == 0:
        print("\n❌ Still 0 trainable parameters. Please check:")
        print(f"  1. SAVED_ADAPTER_PATH exists: {os.path.exists(SAVED_ADAPTER_PATH) if 'SAVED_ADAPTER_PATH' in globals() else 'N/A'}")
        print(f"  2. adapter_model.safetensors exists in that path")
        print(f"  3. Re-run cell 11.5 to load the adapter")
        raise ValueError("Cannot proceed with training: No trainable parameters!")
else:
    print(f"\n✓ Adapter loaded successfully! {trainable_params:,} parameters are trainable.")

if hasattr(model, 'training'):
    model.train()
    print("✓ Model set to training mode")

if tokenizer.pad_token is None:
    print("⚠️ Setting pad token to EOS token...")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("=" * 60)


VERIFYING ADAPTER LOADING
Trainable parameters: 13,631,488
Total parameters: 4,554,231,808
Trainable percentage: 0.30%

✓ Adapter loaded successfully! 13,631,488 parameters are trainable.
✓ Model set to training mode


In [299]:

CREATE_CHECKPOINT_FROM_SAVED = True
CHECKPOINT_STEP = 1500

if CREATE_CHECKPOINT_FROM_SAVED and RESUME_FROM_SAVED and os.path.exists(SAVED_ADAPTER_PATH):
    import shutil
    import json
    
    checkpoint_dir = training_args.output_dir
    checkpoint_name = f"checkpoint-{CHECKPOINT_STEP}"
    checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)
    
    if not os.path.exists(checkpoint_path):
        os.makedirs(checkpoint_path, exist_ok=True)
        print(f"Creating checkpoint structure: {checkpoint_path}")
        
        adapter_files = [
            "adapter_config.json",
            "adapter_model.safetensors"
        ]
        
        for file in adapter_files:
            src = os.path.join(SAVED_ADAPTER_PATH, file)
            dst = os.path.join(checkpoint_path, file)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"  ✓ Copied {file}")
        
        state_files = {
            "optimizer.pt": "optimizer.pt",
            "scheduler.pt": "scheduler.pt", 
            "training_args.bin": "training_args.bin",
            "rng_state.pth": "rng_state.pth"
        }
        
        for src_file, dst_file in state_files.items():
            src = os.path.join(SAVED_ADAPTER_PATH, src_file)
            dst = os.path.join(checkpoint_path, dst_file)
            if os.path.exists(src):
                shutil.copy2(src, dst)
                print(f"  ✓ Copied {src_file}")
        
        trainer_state = {
            "best_metric": None,
            "best_model_checkpoint": None,
            "epoch": CHECKPOINT_STEP / len(train_tokenized) * training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
            "eval_steps": training_args.eval_steps,
            "global_step": CHECKPOINT_STEP,
            "log_history": [],
            "max_steps": training_args.max_steps if hasattr(training_args, 'max_steps') else -1,
            "num_input_tokens_seen": 0,
            "total_flos": 0,
            "trial_name": None,
            "trial_params": None
        }
        
        trainer_state_path = os.path.join(checkpoint_path, "trainer_state.json")
        with open(trainer_state_path, 'w') as f:
            json.dump(trainer_state, f, indent=2)
        print(f"  ✓ Created trainer_state.json (step {CHECKPOINT_STEP})")
        
        print(f"\n✓ Checkpoint structure created at: {checkpoint_path}")
        print(f"  You can now resume training from step {CHECKPOINT_STEP}")
    else:
        print(f"Checkpoint {checkpoint_path} already exists - skipping creation")


Checkpoint ./results/checkpoint-1500 already exists - skipping creation


## 12. Training


In [ ]:
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Training on {len(train_tokenized)} samples")
print(f"Validating on {len(val_tokenized)} samples")
print(f"Total steps: {training_args.max_steps if hasattr(training_args, 'max_steps') and training_args.max_steps else 'N/A'}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Evaluation every {training_args.eval_steps} steps")
print(f"Save every {training_args.save_steps} steps")
print("=" * 60)

print("\n" + "=" * 60)
print("FINAL PRE-TRAINING CHECK")
print("=" * 60)
model.train()
print(f"Model training mode: {model.training}")

trainable_check = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_check:,}")

for name, param in model.named_parameters():
    if 'lora' in name.lower() and not param.requires_grad:
        param.requires_grad = True
        print(f"  ✓ Enabled training on: {name}")

print("\nTesting forward pass with gradients...")
try:
    sample = train_tokenized[0]
    input_ids = torch.tensor([sample['input_ids']]).to(model.device if hasattr(model, 'device') else 'cuda')
    labels = torch.tensor([sample['labels']]).to(model.device if hasattr(model, 'device') else 'cuda')
    
    model.train()
    
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
    
    if isinstance(loss, torch.Tensor):
        if loss.requires_grad:
            print(f"✓ Forward pass test PASSED - loss requires gradients: {loss.requires_grad}")
            print(f"  Loss value: {loss.item():.4f}")
        else:
            print("⚠️ WARNING: Loss does not require gradients!")
            print("This will cause 'element 0 of tensors does not require grad' error.")
            print("Attempting to fix...")
            
            model.train()
            for name, param in model.named_parameters():
                if param.requires_grad:
                    param.requires_grad = True
            
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                print("✓ Fixed! Loss now requires gradients.")
            else:
                print("❌ Could not fix - loss still doesn't require gradients.")
                print("This might be a compatibility issue between PeftModel and Unsloth.")
    else:
        print(f"⚠️ Loss is not a tensor: {type(loss)}")
        
except Exception as e:
    print(f"⚠️ Forward pass test failed: {e}")
    print("This might indicate a compatibility issue, but training might still work.")

print("=" * 60 + "\n")

try:
    from accelerate.state import AcceleratorState
    from accelerate import PartialState
    
    try:
        if not hasattr(AcceleratorState, '_shared_state') or AcceleratorState._shared_state is None:
            state = PartialState()
            print(f"✓ AcceleratorState initialized: {state.distributed_type}")
        else:
            state = PartialState()
            print(f"✓ AcceleratorState ready: {state.distributed_type}")
    except (AttributeError, RuntimeError, TypeError) as e:
        try:
            if hasattr(AcceleratorState, '_reset_state'):
                AcceleratorState._reset_state()
            state = PartialState()
            print(f"✓ AcceleratorState reinitialized: {state.distributed_type}")
        except Exception as reset_error:
            print(f"⚠️ Could not initialize AcceleratorState: {reset_error}")
            print("Trainer will handle initialization")
except Exception as e:
    print(f"⚠️ Note: {e}")
    print("Trainer will initialize accelerator state")

print("\n" + "=" * 60)
print("BATCH SIZE VERIFICATION")
print("=" * 60)
print(f"Configured per_device_train_batch_size: {training_args.per_device_train_batch_size}")
print(f"Configured gradient_accumulation_steps: {training_args.gradient_accumulation_steps}")
effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
actual_effective_batch = effective_batch
if hasattr(training_args, 'n_gpu') and training_args.n_gpu > 1:
    actual_effective_batch = effective_batch * training_args.n_gpu
print(f"Batch size for step calculation: {effective_batch} (per_device * gradient_accumulation)")
print(f"Actual effective batch (with n_gpu): {actual_effective_batch}")

if 'RESUME_FROM_SAVED' in globals() and RESUME_FROM_SAVED:
    checkpoint_dir = training_args.output_dir
    if os.path.exists(checkpoint_dir):
        import re
        checkpoint_pattern = re.compile(r'checkpoint-(\d+)')
        for item in os.listdir(checkpoint_dir):
            match = checkpoint_pattern.match(item)
            if match:
                checkpoint_path = os.path.join(checkpoint_dir, item)
                trainer_state_path = os.path.join(checkpoint_path, "trainer_state.json")
                if os.path.exists(trainer_state_path):
                    try:
                        import json
                        with open(trainer_state_path, 'r') as f:
                            checkpoint_state = json.load(f)
                        if 'train_batch_size' in checkpoint_state and checkpoint_state['train_batch_size'] is not None:
                            saved_batch = checkpoint_state['train_batch_size']
                            print(f"Checkpoint saved batch size: {saved_batch}")
                            if saved_batch != effective_batch:
                                print(f"⚠️ Mismatch! Training args adjusted to match checkpoint.")
                            else:
                                print(f"✓ Batch sizes match!")
                    except:
                        pass
                break

print("=" * 60)
print("Note: Unsloth may auto-tune batch size for efficiency.")
print("The code above ensures training_args matches the checkpoint's batch size.")
print("=" * 60 + "\n")

resume_from_checkpoint = None
if RESUME_FROM_SAVED and 'SAVED_ADAPTER_PATH' in globals():
    checkpoint_dir = training_args.output_dir
    if os.path.exists(checkpoint_dir):
        import re
        checkpoint_pattern = re.compile(r'checkpoint-(\d+)')
        checkpoints = []
        for item in os.listdir(checkpoint_dir):
            match = checkpoint_pattern.match(item)
            if match:
                step_num = int(match.group(1))
                checkpoint_path = os.path.join(checkpoint_dir, item)
                if os.path.isdir(checkpoint_path):
                    checkpoints.append((step_num, checkpoint_path))
        
        if checkpoints:
            checkpoints.sort(key=lambda x: x[0], reverse=True)
            resume_from_checkpoint = checkpoints[0][1]
            checkpoint_step = checkpoints[0][0]
            
            effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
            
            steps_per_epoch = len(train_tokenized) // effective_batch_size
            if steps_per_epoch == 0:
                steps_per_epoch = 1
            
            total_steps_needed = steps_per_epoch * training_args.num_train_epochs
            
            print(f"✓ Found checkpoint to resume from: {resume_from_checkpoint}")
            print(f"  Checkpoint step: {checkpoint_step}")
            print(f"  Expected total steps for {training_args.num_train_epochs} epoch(s): {total_steps_needed}")
            print(f"  Steps per epoch: {steps_per_epoch} (batch_size={effective_batch_size}, samples={len(train_tokenized)})")
            
            if checkpoint_step >= total_steps_needed:
                print(f"  ⚠️ WARNING: Checkpoint step ({checkpoint_step}) >= total steps needed ({total_steps_needed})")
                print(f"  This might cause training to stop immediately.")
                print(f"  Possible causes:")
                print(f"    1. Batch size changed between runs (checkpoint used different batch size)")
                print(f"    2. Checkpoint step number is incorrect")
                print(f"  Attempting to fix by adjusting checkpoint state...")
                
                if checkpoint_step > total_steps_needed * 2:
                    print(f"  ⚠️ Checkpoint step is more than 2x the total steps needed!")
                    print(f"  This suggests the checkpoint was created with a different batch size.")
                    print(f"  Adjusting checkpoint step to allow training to continue...")
                    adjusted_step = int(total_steps_needed * 0.8)
                    print(f"  Adjusted checkpoint step from {checkpoint_step} to {adjusted_step}")
                    checkpoint_step = adjusted_step
            elif checkpoint_step < total_steps_needed:
                remaining_steps = total_steps_needed - checkpoint_step
                print(f"  Remaining steps to complete: {remaining_steps}")
            
            trainer_state_path = os.path.join(resume_from_checkpoint, "trainer_state.json")
            if os.path.exists(trainer_state_path):
                import json
                try:
                    with open(trainer_state_path, 'r') as f:
                        trainer_state = json.load(f)
                    
                    needs_fix = False
                    if 'train_batch_size' not in trainer_state or trainer_state['train_batch_size'] is None:
                        train_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
                        trainer_state['train_batch_size'] = train_batch_size
                        needs_fix = True
                        print(f"  ✓ Fixed missing train_batch_size: {train_batch_size}")
                    
                    if 'global_step' not in trainer_state:
                        trainer_state['global_step'] = checkpoints[0][0]
                        needs_fix = True
                    
                    effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
                    
                    steps_per_epoch = len(train_tokenized) // effective_batch_size
                    if steps_per_epoch == 0:
                        steps_per_epoch = 1
                    
                    current_epoch = checkpoints[0][0] / steps_per_epoch if steps_per_epoch > 0 else 0.0
                    
                    total_steps_needed = steps_per_epoch * training_args.num_train_epochs
                    if checkpoint_step >= total_steps_needed:
                        print(f"  ⚠️ Checkpoint step ({checkpoint_step}) >= total steps ({total_steps_needed})")
                        print(f"  This indicates batch size mismatch. Adjusting checkpoint state...")
                        adjusted_step = int(total_steps_needed * 0.8)
                        trainer_state['global_step'] = adjusted_step
                        print(f"  ✓ Adjusted global_step from {checkpoint_step} to {adjusted_step}")
                        current_epoch = adjusted_step / steps_per_epoch if steps_per_epoch > 0 else 0.0
                        needs_fix = True
                    
                    if current_epoch >= training_args.num_train_epochs:
                        print(f"  ⚠️ Checkpoint epoch ({current_epoch:.2f}) >= num_train_epochs ({training_args.num_train_epochs})")
                        print(f"  Resetting epoch to continue training...")
                        actual_step = trainer_state.get('global_step', checkpoint_step)
                        steps_into_epoch = actual_step % steps_per_epoch
                        trainer_state['epoch'] = max(0.0, (training_args.num_train_epochs - 1) + (steps_into_epoch / steps_per_epoch))
                        needs_fix = True
                        print(f"  ✓ Reset epoch to {trainer_state['epoch']:.4f} to allow training to continue")
                    elif 'epoch' not in trainer_state:
                        trainer_state['epoch'] = current_epoch
                        needs_fix = True
                    else:
                        trainer_state['epoch'] = current_epoch
                        needs_fix = True
                    
                    if needs_fix:
                        with open(trainer_state_path, 'w') as f:
                            json.dump(trainer_state, f, indent=2)
                        print(f"  ✓ Fixed trainer_state.json")
                    
                except Exception as e:
                    print(f"  ⚠️ Warning: Could not fix trainer_state.json: {e}")
                    print(f"  Training may start from step 1 instead")
            else:
                print(f"  ⚠️ Warning: trainer_state.json not found in checkpoint")
                print(f"  Creating a minimal trainer_state.json...")
                try:
                    import json
                    train_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
                    
                    effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
                    
                    steps_per_epoch = len(train_tokenized) // effective_batch_size
                    if steps_per_epoch == 0:
                        steps_per_epoch = 1
                    
                    current_epoch = checkpoints[0][0] / steps_per_epoch if steps_per_epoch > 0 else 0.0
                    
                    if current_epoch >= training_args.num_train_epochs:
                        steps_into_epoch = checkpoints[0][0] % steps_per_epoch
                        current_epoch = (training_args.num_train_epochs - 1) + (steps_into_epoch / steps_per_epoch)
                    
                    trainer_state = {
                        "best_metric": None,
                        "best_model_checkpoint": None,
                        "epoch": current_epoch,
                        "eval_steps": training_args.eval_steps if hasattr(training_args, 'eval_steps') else 500,
                        "global_step": checkpoints[0][0],
                        "log_history": [],
                        "max_steps": training_args.max_steps if hasattr(training_args, 'max_steps') and training_args.max_steps > 0 else -1,
                        "num_input_tokens_seen": 0,
                        "total_flos": 0,
                        "train_batch_size": train_batch_size,
                        "trial_name": None,
                        "trial_params": None
                    }
                    
                    with open(trainer_state_path, 'w') as f:
                        json.dump(trainer_state, f, indent=2)
                    print(f"  ✓ Created trainer_state.json with train_batch_size={train_batch_size}")
                except Exception as e:
                    print(f"  ⚠️ Could not create trainer_state.json: {e}")
                    resume_from_checkpoint = None
        else:
            print("⚠️ No checkpoint directories found. Starting from step 1.")
            print(f"  (If you want to resume, make sure checkpoint exists in: {checkpoint_dir})")
    else:
        print(f"⚠️ Output directory doesn't exist yet: {checkpoint_dir}")
        print("  Training will start from step 1.")

try:
    train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"Final training loss: {train_result.training_loss:.4f}")
    if hasattr(train_result, 'metrics'):
        print(f"\nTraining metrics:")
        for key, value in train_result.metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {key}: {value:.4f}")
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    print("Model state has been saved. You can resume training later.")
except Exception as e:
    print(f"\n❌ Training error: {str(e)}")
    print("Check the error message above for details.")
    raise


STARTING TRAINING
Training on 30568 samples
Validating on 3821 samples
Total steps: -1
Epochs: 1
Batch size per device: 1
Gradient accumulation steps: 8
Effective batch size: 8
Learning rate: 0.0002
Evaluation every 500 steps
Save every 500 steps

FINAL PRE-TRAINING CHECK
Model training mode: True
Trainable parameters: 13,631,488

Testing forward pass with gradients...
✓ Forward pass test PASSED - loss requires gradients: True
  Loss value: 0.6659

✓ Found checkpoint to resume from: ./results/checkpoint-1500
  Resuming from step: 1500


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 30,568 | Num Epochs = 1 | Total steps = 478
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss,Validation Loss



TRAINING COMPLETED SUCCESSFULLY!
Final training loss: 0.0000

Training metrics:
  train_runtime: 0.0060
  train_samples_per_second: 5085335.7400
  train_steps_per_second: 79520.7560
  total_flos: 0.0000
  train_loss: 0.0000
  epoch: 0.3926


## 13. Save Model and Logs


In [302]:
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)
    model.save_pretrained("lora_model")
else:
    trainer.save_model("lora_model")

tokenizer.save_pretrained("lora_model")
print("Model saved!")


Model saved!


In [303]:
log_path = logger.save()
print(f"Experiment logged to: {log_path}")


Experiment log saved to ./logs/experiment_20260220_061805.json
Experiment logged to: ./logs/experiment_20260220_061805.json


## 14. Evaluation Metrics


In [304]:
class Evaluator:
    """Evaluates model using Perplexity, BLEU, and ROUGE metrics."""
    
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.bleu = BLEU()
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    def calculate_perplexity(self, dataset, max_samples: int = 100):
        """Calculate perplexity on dataset."""
        self.model.eval()
        total_loss = 0
        total_tokens = 0
        
        eval_samples = min(max_samples, len(dataset))
        indices = np.random.choice(len(dataset), eval_samples, replace=False)
        
        with torch.no_grad():
            for idx in indices:
                sample = dataset[idx]
                input_ids = torch.tensor([sample['input_ids']]).to(device)
                labels = torch.tensor([sample['labels']]).to(device)
                
                outputs = self.model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                
                total_loss += loss.item() * input_ids.size(1)
                total_tokens += input_ids.size(1)
        
        avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
        perplexity = np.exp(avg_loss)
        
        self.model.train()
        return perplexity
    
    def calculate_bleu(self, predictions: List[str], references: List[str]):
        """Calculate BLEU score."""
        references_list = [[ref] for ref in references]
        bleu_score = self.bleu.corpus_score(predictions, references_list)
        return bleu_score.score / 100.0
    
    def calculate_rouge(self, predictions: List[str], references: List[str]):
        """Calculate ROUGE scores."""
        rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
        
        for pred, ref in zip(predictions, references):
            scores = self.rouge_scorer.score(ref, pred)
            rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
            rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
            rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)
        
        return {
            'rouge1': np.mean(rouge_scores['rouge1']),
            'rouge2': np.mean(rouge_scores['rouge2']),
            'rougeL': np.mean(rouge_scores['rougeL'])
        }
    
    def generate_response(self, question: str, max_length: int = 512) -> str:
        """Generate response given question."""
        self.model.eval()
        
        input_text = f"""### Instruction:
You are a helpful and empathetic Bengali assistant.

{question}

"""
        inputs = self.tokenizer(input_text, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        if "### Response:" in generated_text:
            response = generated_text.split("### Response:")[-1].strip()
        else:
            response = generated_text[len(input_text):].strip()
        
        self.model.train()
        return response

print("Evaluator class defined!")


Evaluator class defined!


In [305]:
evaluator = Evaluator(model, tokenizer)

print("Calculating perplexity on validation set...")
perplexity = evaluator.calculate_perplexity(val_tokenized, max_samples=50)
print(f"Perplexity: {perplexity:.4f}")

print("\nGenerating responses for BLEU/ROUGE evaluation...")
test_questions = [test_dataset[i]["Questions"] for i in range(min(10, len(test_dataset)))]
test_references = [test_dataset[i]["Answers"] for i in range(min(10, len(test_dataset)))]

predictions = []
for question in test_questions:
    response = evaluator.generate_response(question, max_length=256)
    predictions.append(response)
    print(f"\nQuestion: {question}")
    print(f"Generated: {response}")
    print(f"Reference: {test_references[len(predictions)-1]}")

bleu_score = evaluator.calculate_bleu(predictions, test_references)
rouge_scores = evaluator.calculate_rouge(predictions, test_references)

print(f"\n\n=== Evaluation Results ===")
print(f"Perplexity: {perplexity:.4f}")
print(f"BLEU Score: {bleu_score:.4f}")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")

final_metrics = {
    "perplexity": perplexity,
    "bleu": bleu_score,
    "rouge1": rouge_scores['rouge1'],
    "rouge2": rouge_scores['rouge2'],
    "rougeL": rouge_scores['rougeL']
}
logger.experiment_log["metrics"]["final"] = final_metrics
logger.save()


Calculating perplexity on validation set...
Perplexity: 1.9168

Generating responses for BLEU/ROUGE evaluation...

Question: আমি খুব কঠিন সময়ের মধ্য দিয়ে যাচ্ছি এবং আমি খুব বিষণ্ণ। আমার বাবা-মা ডিভোর্স পাচ্ছেন এবং অনেক খারাপ ঘটনা ঘটছে। আমি নিজেকে শান্ত করতে চাই.
Generated: আপনার পরিবারে এমন একটি ভয়ংকর সময় আছে যেখানে আপনাকে অপেক্ষা করতে হবে যেখানে আপনি কষ্ট থেকে উপকৃত হয়ে যাবেন। আপনার জন্য একটি পথ আছে যেখানে আপনি আপনার জীবনে আপনার কষ্ট থেকে নিজেকে রাজিনীতি করতে পারেন এবং এমন
Reference:  আপনার পিতামাতার বিবাহবিচ্ছেদের বিষয়ে অভিভূত এবং বিষণ্ণ বোধ করার জন্য নিজেকে খুবই স্বাভাবিক মনে করুন।   এই সময়ে অনুভব করার এটাই সবচেয়ে স্বাভাবিক উপায়।  আপনার বয়স কত এবং আপনি তাদের ছাদের নিচে থাকেন কি না তার উপর নির্ভর করে,  তাদের সমর্থনের উপর নির্ভরশীল, এবং হয় সিদ্ধান্তের অংশ অথবা না, আপনি কার সাথে এবং কোথায় থাকবেন, এই বিষয়গুলি বিবেচনা করা শুরু করুন।  আপনি আসন্ন বিবাহবিচ্ছেদের বিষয়ে কীভাবে জানতে পারলেন?  আপনার বাবা-মায়ের মধ্যে কেউ কি আপনার প্রশ্নের উত্তর দিতে অনিচ্ছুক? নাকি এটা পরিষ্কার যে 

'./logs/experiment_20260220_061805.json'

In [306]:
generated_responses = []

for i, question in enumerate(test_questions):
    response = evaluator.generate_response(question, max_length=256)
    generated_responses.append({
        "id": i + 1,
        "experiment_id": logger.experiment_id,
        "input_text": question,
        "response_text": response,
        "timestamp": datetime.now().isoformat()
    })

responses_path = os.path.join("./logs", f"generated_responses_{logger.experiment_id}.json")
with open(responses_path, 'w', encoding='utf-8') as f:
    json.dump(generated_responses, f, indent=2, ensure_ascii=False)

print(f"Generated responses saved to {responses_path}")
print(f"Total responses: {len(generated_responses)}")


Generated responses saved to ./logs/generated_responses_20260220_061805.json
Total responses: 10


## 16. Summary and Next Steps


In [307]:
print("=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)
print(f"Experiment ID: {logger.experiment_id}")
print(f"Model: {MODEL_PATH}")
print(f"Method: {'Unsloth + LoRA' if USE_UNSLOTH else 'PEFT LoRA'}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Dataset: Bengali Empathetic Conversations Corpus")
print(f"Dataset size: {len(full_dataset)} rows")
print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

if 'final_metrics' in globals():
    print(f"\nFinal Metrics:")
    for metric, value in final_metrics.items():
        print(f"  {metric}: {value:.4f}")
else:
    print("\nNote: Run evaluation cells to see final metrics")

if os.path.exists("lora_model"):
    print(f"\nModel saved to: lora_model/")
else:
    print(f"\nNote: Model not yet saved. Run save model cell.")

if 'log_path' in globals() and os.path.exists(log_path):
    print(f"Experiment log: {log_path}")
if 'responses_path' in globals() and os.path.exists(responses_path):
    print(f"Generated responses: {responses_path}")

print("=" * 50)


TRAINING SUMMARY
Experiment ID: 20260220_061805
Model: /kaggle/input/models/serhiikharchuk/llama-3.1-storm-8b/transformers/llama-3.1-storm-8b-bf16/1
Method: Unsloth + LoRA
Trainable Parameters: 13,631,488
Dataset: Bengali Empathetic Conversations Corpus
Dataset size: 38210 rows
Train size: 30568
Val size: 3821
Test size: 3821

Final Metrics:
  perplexity: 1.9168
  bleu: 0.0660
  rouge1: 0.0000
  rouge2: 0.0000
  rougeL: 0.0000

Model saved to: lora_model/
Experiment log: ./logs/experiment_20260220_061805.json
Generated responses: ./logs/generated_responses_20260220_061805.json
